<a href="https://colab.research.google.com/github/eya-sheikhrouhou/Ipssi-Projects/blob/API/Mini_Projet2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini-projet API — PokéAPI
La PokéAPI (https://pokeapi.co/) est une API REST gratuite, sans clé, qui fournit des informations
sur les Pokémon, leurs types, leurs évolutions, etc.
Endpoints principaux :
- /api/v2/pokemon/{id ou nom} : informations sur un Pokémon
- /api/v2/type/{type} : informations sur un type (fire, water, dragon...)

## 1. Exploration de la réponse de l'API
On interroge /pokemon/pikachu pour observer la structure du JSON renvoyé.

In [1]:
import requests

BASE_URL = "https://pokeapi.co/api/v2"

response = requests.get(f"{BASE_URL}/pokemon/pikachu", timeout=10)
print("Code de réponse :", response.status_code)

data = response.json()

# On affiche seulement les clés du dictionnaire, pas tout le contenu
print("\nClés disponibles :")
print(list(data.keys()))

Code de réponse : 200

Clés disponibles :
['id', 'name', 'base_experience', 'height', 'is_default', 'order', 'weight', 'abilities', 'past_abilities', 'forms', 'game_indices', 'held_items', 'location_area_encounters', 'moves', 'species', 'sprites', 'cries', 'stats', 'past_stats', 'types', 'past_types']


In [2]:
print("Nom :", data["name"])
print("Numéro :", data["id"])
print("Taille :", data["height"])
print("Poids :", data["weight"])

print("\nNombre d'attaques apprises :", len(data["moves"]), "(c'est pour ça que le JSON est énorme)")

print("\nTypes :")
print(data["types"])

print("\nStatistiques :")
for stat in data["stats"]:
    print(stat)

Nom : pikachu
Numéro : 25
Taille : 4
Poids : 60

Nombre d'attaques apprises : 109 (c'est pour ça que le JSON est énorme)

Types :
[{'slot': 1, 'type': {'name': 'electric', 'url': 'https://pokeapi.co/api/v2/type/13/'}}]

Statistiques :
{'base_stat': 35, 'effort': 0, 'stat': {'name': 'hp', 'url': 'https://pokeapi.co/api/v2/stat/1/'}}
{'base_stat': 55, 'effort': 0, 'stat': {'name': 'attack', 'url': 'https://pokeapi.co/api/v2/stat/2/'}}
{'base_stat': 40, 'effort': 0, 'stat': {'name': 'defense', 'url': 'https://pokeapi.co/api/v2/stat/3/'}}
{'base_stat': 50, 'effort': 0, 'stat': {'name': 'special-attack', 'url': 'https://pokeapi.co/api/v2/stat/4/'}}
{'base_stat': 50, 'effort': 0, 'stat': {'name': 'special-defense', 'url': 'https://pokeapi.co/api/v2/stat/5/'}}
{'base_stat': 90, 'effort': 2, 'stat': {'name': 'speed', 'url': 'https://pokeapi.co/api/v2/stat/6/'}}


## 2. Afficher les statistiques d'un Pokémon
- get_pokemon(nom_ou_id) : interroge l'API et ne garde que les informations utiles
  (nom, numéro, types, taille, poids, statistiques de base) ;
- afficher_stats_pokemon() : demande un Pokémon à l'utilisateur et affiche ses statistiques.
Taille convertie de décimètres en mètres et poids d'hectogrammes en kilos.

In [7]:
# Traduction des noms de statistiques pour l'affichage
NOMS_STATS = {
    "hp": "Points de vie",
    "attack": "Attaque",
    "defense": "Défense",
    "special-attack": "Attaque spéciale",
    "special-defense": "Défense spéciale",
    "speed": "Vitesse",
}

def get_pokemon(nom_ou_id):
    """Récupère un Pokémon et renvoie un dictionnaire avec seulement les infos utiles.
    Renvoie None si le Pokémon n'existe pas."""

    # L'API attend un nom en minuscules, sans espaces autour
    nom_ou_id = str(nom_ou_id).strip().lower()

    response = requests.get(f"{BASE_URL}/pokemon/{nom_ou_id}", timeout=10)
    if response.status_code != 200:
        return None

    data = response.json()

    # On transforme la liste des stats en dictionnaire simple : {"hp": 35, "attack": 55, ...}
    stats = {s["stat"]["name"]: s["base_stat"] for s in data["stats"]}

    return {
        "nom": data["name"],
        "id": data["id"],
        "types": [t["type"]["name"] for t in data["types"]],
        "taille_m": data["height"] / 10,     # décimètres → mètres
        "poids_kg": data["weight"] / 10,     # hectogrammes → kilos
        "stats": stats,
    }


# Test
get_pokemon("pikachu")

{'nom': 'pikachu',
 'id': 25,
 'types': ['electric'],
 'taille_m': 0.4,
 'poids_kg': 6.0,
 'stats': {'hp': 35,
  'attack': 55,
  'defense': 40,
  'special-attack': 50,
  'special-defense': 50,
  'speed': 90}}

In [9]:
stats = {s["stat"]["name"]: s["base_stat"] for s in data["stats"]}
stats

{'hp': 35,
 'attack': 55,
 'defense': 40,
 'special-attack': 50,
 'special-defense': 50,
 'speed': 90}

In [10]:
def afficher_stats_pokemon():
    """Demande un Pokémon à l'utilisateur et affiche ses statistiques."""

    choix = input("Nom ou numéro du Pokémon : ")
    pokemon = get_pokemon(choix)

    if pokemon is None:
        print(f"❌ Pokémon « {choix} » introuvable. Vérifie l'orthographe (nom anglais, ex. : charizard).")
        return

    print(f"\n=== {pokemon['nom'].capitalize()} (n°{pokemon['id']}) ===")
    print(f"Type(s) : {', '.join(pokemon['types'])}")
    print(f"Taille  : {pokemon['taille_m']} m   |   Poids : {pokemon['poids_kg']} kg\n")

    for nom_stat, valeur in pokemon["stats"].items():
        barre = "█" * (valeur // 10)
        print(f"{NOMS_STATS[nom_stat]:<18} {valeur:>4}  {barre}")

    print(f"{'Total':<18} {sum(pokemon['stats'].values()):>4}")


afficher_stats_pokemon()

Nom ou numéro du Pokémon : Pikachu

=== Pikachu (n°25) ===
Type(s) : electric
Taille  : 0.4 m   |   Poids : 6.0 kg

Points de vie        35  ███
Attaque              55  █████
Défense              40  ████
Attaque spéciale     50  █████
Défense spéciale     50  █████
Vitesse              90  █████████
Total               320


## 3. Comparer deux Pokémon
La fonction demande deux Pokémon, affiche leurs statistiques côte à côte
et indique lequel est le plus fort en points de vie et en attaque (égalité gérée).
Elle réutilise get_pokemon() de l'étape 2.

In [11]:
def gagnant(pokemon1, pokemon2, nom_stat):
    """Renvoie le nom du Pokémon qui a la plus grande valeur pour une statistique,
    ou 'Égalité' s'ils ont la même."""
    valeur1 = pokemon1["stats"][nom_stat]
    valeur2 = pokemon2["stats"][nom_stat]

    if valeur1 > valeur2:
        return pokemon1["nom"].capitalize()
    elif valeur2 > valeur1:
        return pokemon2["nom"].capitalize()
    else:
        return "Égalité"

In [12]:
def comparer_pokemons():
    """Demande deux Pokémon et compare leurs statistiques."""

    choix1 = input("Premier Pokémon : ")
    choix2 = input("Deuxième Pokémon : ")

    pokemon1 = get_pokemon(choix1)
    pokemon2 = get_pokemon(choix2)

    # Vérifier que les deux existent
    if pokemon1 is None:
        print(f"❌ Pokémon « {choix1} » introuvable.")
        return
    if pokemon2 is None:
        print(f"❌ Pokémon « {choix2} » introuvable.")
        return

    nom1 = pokemon1["nom"].capitalize()
    nom2 = pokemon2["nom"].capitalize()

    # Tableau comparatif
    print(f"\n{'Statistique':<18} {nom1:>12} {nom2:>12}   Meilleur")
    print("-" * 60)
    for nom_stat in pokemon1["stats"]:
        v1 = pokemon1["stats"][nom_stat]
        v2 = pokemon2["stats"][nom_stat]
        print(f"{NOMS_STATS[nom_stat]:<18} {v1:>12} {v2:>12}   {gagnant(pokemon1, pokemon2, nom_stat)}")

    total1 = sum(pokemon1["stats"].values())
    total2 = sum(pokemon2["stats"].values())
    print("-" * 60)
    print(f"{'Total':<18} {total1:>12} {total2:>12}")

    # La réponse à la question de l'énoncé
    print("\n🏆 Verdict :")
    print(f"   Plus fort en points de vie : {gagnant(pokemon1, pokemon2, 'hp')}")
    print(f"   Plus fort en attaque       : {gagnant(pokemon1, pokemon2, 'attack')}")


comparer_pokemons()

Premier Pokémon : charizard
Deuxième Pokémon : pikachu

Statistique           Charizard      Pikachu   Meilleur
------------------------------------------------------------
Points de vie                78           35   Charizard
Attaque                      84           55   Charizard
Défense                      78           40   Charizard
Attaque spéciale            109           50   Charizard
Défense spéciale             85           50   Charizard
Vitesse                     100           90   Charizard
------------------------------------------------------------
Total                       534          320

🏆 Verdict :
   Plus fort en points de vie : Charizard
   Plus fort en attaque       : Charizard


In [13]:
{'pokemon': [
    {'pokemon': {'name': 'charmander', 'url': 'https://pokeapi.co/api/v2/pokemon/4/'}, 'slot': 1},
    {'pokemon': {'name': 'charmeleon', 'url': '...'}, 'slot': 1},
    ...
]}

{'pokemon': [{'pokemon': {'name': 'charmander',
    'url': 'https://pokeapi.co/api/v2/pokemon/4/'},
   'slot': 1},
  {'pokemon': {'name': 'charmeleon', 'url': '...'}, 'slot': 1},
  Ellipsis]}

## 4. Statistiques d'un type
L'endpoint /type/{type} renvoie la liste des Pokémon du type, mais pas leurs statistiques.
- Nombre de Pokémon : longueur de cette liste (1 seule requête).
- Moyenne des points de vie : il faut récupérer chaque Pokémon un par un (une requête par Pokémon),
  ce qui est lent en séquentiel → amélioré en Q2 avec asyncio.

In [14]:
response = requests.get(f"{BASE_URL}/type/fire", timeout=10)
data_type = response.json()

print("Clés :", list(data_type.keys()))
print("\nNombre de Pokémon de type fire :", len(data_type["pokemon"]))
print("\nLes 3 premiers :")
for p in data_type["pokemon"][:3]:
    print(p)

Clés : ['id', 'name', 'damage_relations', 'past_damage_relations', 'game_indices', 'generation', 'move_damage_class', 'names', 'pokemon', 'moves', 'sprites']

Nombre de Pokémon de type fire : 109

Les 3 premiers :
{'slot': 1, 'pokemon': {'name': 'charmander', 'url': 'https://pokeapi.co/api/v2/pokemon/4/'}}
{'slot': 1, 'pokemon': {'name': 'charmeleon', 'url': 'https://pokeapi.co/api/v2/pokemon/5/'}}
{'slot': 1, 'pokemon': {'name': 'charizard', 'url': 'https://pokeapi.co/api/v2/pokemon/6/'}}


In [15]:
import time

def stats_type():
    """Demande un type et calcule le nombre de Pokémon et la moyenne de leurs PV."""

    nom_type = input("Type de Pokémon (ex. fire, water, dragon) : ").strip().lower()

    # 1. Récupérer la liste des Pokémon du type
    response = requests.get(f"{BASE_URL}/type/{nom_type}", timeout=10)
    if response.status_code != 200:
        print(f"❌ Type « {nom_type} » introuvable.")
        return

    noms = [p["pokemon"]["name"] for p in response.json()["pokemon"]]
    nb = len(noms)
    print(f"\n🔥 Nombre de Pokémon de type {nom_type} : {nb}")

    if nb == 0:
        print("Aucun Pokémon pour ce type, impossible de calculer une moyenne.")
        return

    # 2. Récupérer chaque Pokémon pour avoir ses PV (une requête par Pokémon)
    print(f"⏳ Récupération des {nb} Pokémon un par un...")
    debut = time.perf_counter()

    pv = {}
    for i, nom in enumerate(noms, start=1):
        pokemon = get_pokemon(nom)
        if pokemon is not None:
            pv[nom] = pokemon["stats"]["hp"]
        if i % 20 == 0:
            print(f"   {i}/{nb}...")

    duree = time.perf_counter() - debut

    # 3. Calcul de la moyenne
    moyenne = sum(pv.values()) / len(pv)

    print(f"\n❤️ Moyenne des points de vie : {moyenne:.1f}")
    print(f"   PV minimum : {min(pv.values())} ({min(pv, key=pv.get)})")
    print(f"   PV maximum : {max(pv.values())} ({max(pv, key=pv.get)})")
    print(f"\n⏱️ Temps de calcul : {duree:.1f} s pour {nb} requêtes (séquentiel)")


stats_type()

Type de Pokémon (ex. fire, water, dragon) : fire

🔥 Nombre de Pokémon de type fire : 109
⏳ Récupération des 109 Pokémon un par un...
   20/109...
   40/109...
   60/109...
   80/109...
   100/109...

❤️ Moyenne des points de vie : 72.5
   PV minimum : 38 (vulpix)
   PV maximum : 115 (entei)

⏱️ Temps de calcul : 7.8 s pour 109 requêtes (séquentiel)


## 5. Q1 – Gestion des erreurs courantes
On réutilise la fonction générale requete_robuste (écrite pour le projet TMDB) :
- erreurs temporaires (429, 500, 502, 503, 504, timeout, réseau) : pause puis nouvelle tentative,
  avec un délai qui double à chaque essai (backoff exponentiel) ;
- erreurs définitives (400, 401, 403, 404) : arrêt immédiat avec un message clair.
On ajoute une validation du nom/identifiant avant l'appel, puis get_pokemon est réécrite pour
utiliser requete_robuste : les fonctions des étapes 2 et 3 deviennent robustes sans modification.

In [16]:
import time
import requests

CODES_TEMPORAIRES = {429, 500, 502, 503, 504}

MESSAGES_ERREURS = {
    400: "Requête mal formée (400) : vérifie les paramètres envoyés",
    401: "Non autorisé (401) : authentification absente ou invalide",
    403: "Accès interdit (403) : tu n'as pas les droits sur cette ressource",
    404: "Introuvable (404) : cet élément n'existe pas",
}

def requete_robuste(url, params=None, max_tentatives=5, pause_initiale=1, timeout=10):
    """Envoie une requête GET en gérant les erreurs courantes.
    Renvoie le JSON de la réponse, ou None en cas d'échec."""

    for tentative in range(1, max_tentatives + 1):
        attente = pause_initiale * 2 ** (tentative - 1)

        try:
            response = requests.get(url, params=params, timeout=timeout)
        except requests.exceptions.Timeout:
            print(f"⏳ Tentative {tentative} : pas de réponse (timeout), nouvel essai dans {attente}s")
            time.sleep(attente)
            continue
        except requests.exceptions.ConnectionError:
            print(f"⏳ Tentative {tentative} : problème de connexion, nouvel essai dans {attente}s")
            time.sleep(attente)
            continue

        code = response.status_code

        if code == 200:
            return response.json()

        if code in CODES_TEMPORAIRES:
            if code == 429 and "Retry-After" in response.headers:
                attente = int(response.headers["Retry-After"])
            print(f"⏳ Tentative {tentative} : erreur {code}, nouvel essai dans {attente}s")
            time.sleep(attente)
            continue

        print("❌", MESSAGES_ERREURS.get(code, f"Erreur inattendue ({code})"))
        return None

    print(f"❌ Échec après {max_tentatives} tentatives")
    return None

In [17]:
import re

def nettoyer_identifiant(nom_ou_id):
    """Vérifie et nettoie un nom ou un numéro de Pokémon.
    Renvoie la version propre, ou None si l'identifiant n'est pas valide."""

    texte = str(nom_ou_id).strip().lower()

    if texte == "":
        print("❌ Identifiant vide : tape un nom ou un numéro")
        return None

    # Cas d'un numéro : il doit être positif
    if texte.lstrip("-").isdigit():
        if int(texte) <= 0:
            print(f"❌ Numéro non valide : {texte} (il faut un nombre positif)")
            return None
        return texte

    # Cas d'un nom : seulement des lettres minuscules, des chiffres et des tirets
    if not re.fullmatch(r"[a-z0-9-]+", texte):
        print(f"❌ Nom non valide : « {nom_ou_id} » (lettres, chiffres et tirets uniquement)")
        return None

    return texte

In [18]:
def get_pokemon(nom_ou_id):
    """Version robuste : validation + requete_robuste.
    Renvoie un dictionnaire avec les infos utiles, ou None en cas d'erreur."""

    identifiant = nettoyer_identifiant(nom_ou_id)
    if identifiant is None:
        return None

    data = requete_robuste(f"{BASE_URL}/pokemon/{identifiant}")
    if data is None:
        return None

    stats = {s["stat"]["name"]: s["base_stat"] for s in data.get("stats", [])}

    return {
        "nom": data.get("name"),
        "id": data.get("id"),
        "types": [t["type"]["name"] for t in data.get("types", [])],
        "taille_m": data.get("height", 0) / 10,
        "poids_kg": data.get("weight", 0) / 10,
        "stats": stats,
    }


def get_noms_par_type(nom_type):
    """Version robuste : renvoie la liste des noms de Pokémon d'un type, ou None."""
    identifiant = nettoyer_identifiant(nom_type)
    if identifiant is None:
        return None
    data = requete_robuste(f"{BASE_URL}/type/{identifiant}")
    if data is None:
        return None
    return [p["pokemon"]["name"] for p in data.get("pokemon", [])]

In [19]:
print("=== Cas normaux ===")
print(get_pokemon("pikachu"))
print(get_pokemon(25))
print(get_pokemon("  Charizard  "))

print("\n=== Identifiants non valides (aucun appel à l'API) ===")
print(get_pokemon(""))
print(get_pokemon(-5))
print(get_pokemon(0))
print(get_pokemon("pika chu!"))

print("\n=== Pokémon et type inexistants (404) ===")
print(get_pokemon("pikachuuu"))
print(get_noms_par_type("feu"))

print("\n=== Simulation des autres erreurs avec httpbin ===")
for code in [400, 401, 403, 429, 503]:
    print(f"\n--- Test du code {code} ---")
    requete_robuste(f"https://httpbin.org/status/{code}", max_tentatives=3)

=== Cas normaux ===
{'nom': 'pikachu', 'id': 25, 'types': ['electric'], 'taille_m': 0.4, 'poids_kg': 6.0, 'stats': {'hp': 35, 'attack': 55, 'defense': 40, 'special-attack': 50, 'special-defense': 50, 'speed': 90}}
{'nom': 'pikachu', 'id': 25, 'types': ['electric'], 'taille_m': 0.4, 'poids_kg': 6.0, 'stats': {'hp': 35, 'attack': 55, 'defense': 40, 'special-attack': 50, 'special-defense': 50, 'speed': 90}}
{'nom': 'charizard', 'id': 6, 'types': ['fire', 'flying'], 'taille_m': 1.7, 'poids_kg': 90.5, 'stats': {'hp': 78, 'attack': 84, 'defense': 78, 'special-attack': 109, 'special-defense': 85, 'speed': 100}}

=== Identifiants non valides (aucun appel à l'API) ===
❌ Identifiant vide : tape un nom ou un numéro
None
❌ Numéro non valide : -5 (il faut un nombre positif)
None
❌ Numéro non valide : 0 (il faut un nombre positif)
None
❌ Nom non valide : « pika chu! » (lettres, chiffres et tirets uniquement)
None

=== Pokémon et type inexistants (404) ===
❌ Introuvable (404) : cet élément n'existe p

## 6. Q2 – Gérer un très grand nombre de requêtes (asyncio)
Partie A : en séquentiel, chaque requête attend la précédente (≈ 1 min pour un type).
Avec asyncio + aiohttp, les requêtes partent en parallèle : pendant qu'une attend sa réponse,
les autres avancent. Un Semaphore limite à 20 requêtes simultanées pour respecter PokéAPI.
La logique de retry de la Q1 est conservée dans la version asynchrone.
Partie B : simulation d'une charge très élevée (100 000 appels) sur 3 routes fictives
(api.example.com n'existant pas, on simule le serveur).

In [20]:
# On installe aiohttp, la bibliothèque qui permet d'envoyer des requêtes HTTP de façon asynchrone.
!pip install -q aiohttp

In [21]:
# Cette fonction transforme le gros JSON d'un Pokémon en petit dictionnaire propre (réutilisable partout).
def formater_pokemon(data):
    stats = {s["stat"]["name"]: s["base_stat"] for s in data.get("stats", [])}
    return {
        "nom": data.get("name"),
        "id": data.get("id"),
        "types": [t["type"]["name"] for t in data.get("types", [])],
        "taille_m": data.get("height", 0) / 10,
        "poids_kg": data.get("weight", 0) / 10,
        "stats": stats,
    }

In [22]:
# Version asynchrone de requete_robuste (Q1) : même logique de retry, mais sans bloquer les autres requêtes.
import asyncio
import aiohttp

async def requete_robuste_async(session, url, semaphore, max_tentatives=5, pause_initiale=1):
    async with semaphore:                       # on attend qu'une place soit libre
        for tentative in range(1, max_tentatives + 1):
            attente = pause_initiale * 2 ** (tentative - 1)
            try:
                async with session.get(url, timeout=aiohttp.ClientTimeout(total=10)) as response:
                    code = response.status

                    # Succès
                    if code == 200:
                        return await response.json()

                    # Erreur définitive : on arrête
                    if code not in CODES_TEMPORAIRES:
                        print("❌", MESSAGES_ERREURS.get(code, f"Erreur inattendue ({code})"), "→", url)
                        return None

                    # Erreur temporaire : on va réessayer
                    if code == 429 and "Retry-After" in response.headers:
                        attente = int(response.headers["Retry-After"])
                    print(f"⏳ {url} : erreur {code}, nouvel essai dans {attente}s")

            except (aiohttp.ClientError, asyncio.TimeoutError):
                print(f"⏳ {url} : problème réseau, nouvel essai dans {attente}s")

            await asyncio.sleep(attente)        # pause qui ne bloque pas les autres requêtes

        print(f"❌ Échec après {max_tentatives} tentatives → {url}")
        return None

In [23]:
# Version asynchrone de get_pokemon : valide l'identifiant, appelle l'API sans bloquer et renvoie le dictionnaire propre.
async def get_pokemon_async(session, nom_ou_id, semaphore):
    identifiant = nettoyer_identifiant(nom_ou_id)
    if identifiant is None:
        return None
    data = await requete_robuste_async(session, f"{BASE_URL}/pokemon/{identifiant}", semaphore)
    return formater_pokemon(data) if data else None

In [24]:
# Statistiques d'un type en asynchrone : on récupère tous les Pokémon du type en parallèle (20 à la fois).
async def stats_type_async(nom_type, max_simultanees=20):
    noms = get_noms_par_type(nom_type)          # 1 seule requête (version robuste de la Q1)
    if not noms:
        print(f"❌ Aucun Pokémon trouvé pour le type « {nom_type} ».")
        return None

    print(f"🔥 Nombre de Pokémon de type {nom_type} : {len(noms)}")
    print(f"⏳ Récupération en parallèle ({max_simultanees} requêtes simultanées)...")

    semaphore = asyncio.Semaphore(max_simultanees)
    debut = time.perf_counter()

    async with aiohttp.ClientSession() as session:
        taches = [get_pokemon_async(session, nom, semaphore) for nom in noms]
        pokemons = await asyncio.gather(*taches)

    duree = time.perf_counter() - debut

    pv = {p["nom"]: p["stats"]["hp"] for p in pokemons if p is not None}
    moyenne = sum(pv.values()) / len(pv)

    print(f"\n❤️ Moyenne des points de vie : {moyenne:.1f}")
    print(f"   PV minimum : {min(pv.values())} ({min(pv, key=pv.get)})")
    print(f"   PV maximum : {max(pv.values())} ({max(pv, key=pv.get)})")
    print(f"\n⏱️ Temps de calcul : {duree:.1f} s pour {len(noms)} requêtes (asynchrone)")
    return duree

In [25]:
# On demande un type à l'utilisateur et on lance le calcul asynchrone, pour comparer avec le temps de l'étape 4.
nom_type = input("Type de Pokémon (ex. fire, water, dragon) : ")
await stats_type_async(nom_type)

Type de Pokémon (ex. fire, water, dragon) : water
🔥 Nombre de Pokémon de type water : 192
⏳ Récupération en parallèle (20 requêtes simultanées)...

❤️ Moyenne des points de vie : 71.6
   PV minimum : 10 (wiglett)
   PV maximum : 170 (wailord)

⏱️ Temps de calcul : 0.7 s pour 192 requêtes (asynchrone)


0.7295539729998382

In [26]:
# Faux serveur qui imite une API : un temps de réponse aléatoire et quelques erreurs aléatoires (429, 503, 404).
import random

URLS = [
    "https://api.example.com/data1",
    "https://api.example.com/data2",
    "https://api.example.com/data3",
]

async def faux_serveur(url):
    await asyncio.sleep(random.uniform(0.05, 0.2))   # temps de réponse du "réseau"
    tirage = random.random()
    if tirage < 0.05:
        return 429      # 5 % : trop de requêtes
    if tirage < 0.08:
        return 503      # 3 % : serveur indisponible
    if tirage < 0.09:
        return 404      # 1 % : introuvable
    return 200          # le reste : succès

In [27]:
# Version de la requête robuste pour le faux serveur : même retry que la Q1, et un résultat résumé pour les statistiques.
async def requete_simulee_async(url, semaphore, max_tentatives=5, pause_initiale=0.1):
    async with semaphore:
        for tentative in range(1, max_tentatives + 1):
            code = await faux_serveur(url)
            if code == 200:
                return {"url": url, "statut": "succès", "tentatives": tentative}
            if code in CODES_TEMPORAIRES:
                await asyncio.sleep(pause_initiale * 2 ** (tentative - 1))
                continue
            return {"url": url, "statut": f"erreur {code}", "tentatives": tentative}
        return {"url": url, "statut": "échec", "tentatives": max_tentatives}


async def lancer_charge(n, max_simultanees):
    semaphore = asyncio.Semaphore(max_simultanees)
    taches = [requete_simulee_async(URLS[i % 3], semaphore) for i in range(n)]
    return await asyncio.gather(*taches)

In [28]:
# Comparaison sur 100 requêtes : une seule à la fois (séquentiel) contre 1000 à la fois (concurrent).
debut = time.perf_counter()
await lancer_charge(100, max_simultanees=1)
duree_seq = time.perf_counter() - debut
print(f"🐢 Séquentiel : 100 requêtes en {duree_seq:.2f} s")

debut = time.perf_counter()
await lancer_charge(100, max_simultanees=1000)
duree_conc = time.perf_counter() - debut
print(f"🚀 Concurrent : 100 requêtes en {duree_conc:.2f} s")

print(f"➡️ {duree_seq / duree_conc:.0f} fois plus rapide")

🐢 Séquentiel : 100 requêtes en 13.97 s
🚀 Concurrent : 100 requêtes en 0.45 s
➡️ 31 fois plus rapide


In [29]:
# Test de forte charge : 100 000 requêtes réparties sur les 3 routes, avec statistiques détaillées.
from collections import Counter

N = 100_000
debut = time.perf_counter()
resultats = await lancer_charge(N, max_simultanees=1000)
duree = time.perf_counter() - debut

print(f"✅ {N:,} requêtes en {duree:.1f} s, soit {N / duree:,.0f} requêtes par seconde")
print("\nRépartition par statut :", Counter(r["statut"] for r in resultats))
print("Répartition par route  :", Counter(r["url"] for r in resultats))
print("Requêtes ayant nécessité un retry :", sum(r["tentatives"] > 1 for r in resultats))
print(f"\n➡️ Estimation pour 1 million de requêtes : environ {1_000_000 / (N / duree) / 60:.1f} minutes")

✅ 100,000 requêtes en 17.5 s, soit 5,725 requêtes par seconde

Répartition par statut : Counter({'succès': 98907, 'erreur 404': 1093})
Répartition par route  : Counter({'https://api.example.com/data1': 33334, 'https://api.example.com/data2': 33333, 'https://api.example.com/data3': 33333})
Requêtes ayant nécessité un retry : 7968

➡️ Estimation pour 1 million de requêtes : environ 2.9 minutes


## 7. Q3 – Cache avec expiration (TTL = 1 heure)
Le cache garde en mémoire les réponses de l'API. Si la même requête est refaite dans l'heure,
la réponse stockée est renvoyée immédiatement (cache hit) ; au-delà, elle est périmée et l'API
est rappelée (cache miss). Les erreurs ne sont jamais mises en cache.
Avantages : moins d'appels à l'API (moins de charge serveur, moins de risques de 429)
et des réponses beaucoup plus rapides. PokéAPI recommande d'ailleurs de mettre ses réponses en cache.
Le cache est partagé entre les versions synchrone et asynchrone des fonctions.

In [30]:
# Classe qui stocke des réponses en mémoire avec leur heure d'arrivée, et les considère périmées après `duree_vie` secondes.
class CacheExpirant:

    def __init__(self, duree_vie=3600):
        self.duree_vie = duree_vie   # 3600 s = 1 heure
        self.donnees = {}            # { clé : (valeur, heure_de_stockage) }
        self.hits = 0                # nombre de fois où le cache a servi
        self.misses = 0              # nombre de fois où il a fallu appeler l'API

    def get(self, cle):
        """Renvoie la valeur si elle est en cache et pas périmée, sinon None."""
        if cle in self.donnees:
            valeur, heure_stockage = self.donnees[cle]
            if time.time() - heure_stockage < self.duree_vie:
                self.hits += 1
                return valeur
            del self.donnees[cle]    # périmée : on la supprime
        self.misses += 1
        return None

    def set(self, cle, valeur):
        """Stocke une valeur avec l'heure actuelle."""
        self.donnees[cle] = (valeur, time.time())

    def stats(self):
        total = self.hits + self.misses
        taux = 100 * self.hits / total if total else 0
        return f"{self.hits} hits / {self.misses} misses (taux de réussite du cache : {taux:.1f} %)"

In [31]:
# On crée un cache d'1 heure partagé par tout le notebook, et une fonction qui consulte ce cache avant d'appeler l'API.
cache_api = CacheExpirant(duree_vie=3600)

def requete_avec_cache(url, cache=cache_api):
    resultat = cache.get(url)
    if resultat is not None:
        return resultat                 # ⚡ cache hit : pas d'appel à l'API

    resultat = requete_robuste(url)     # cache miss : appel robuste (Q1)
    if resultat is not None:            # on ne garde que les succès, jamais les erreurs
        cache.set(url, resultat)
    return resultat

In [32]:
# On redéfinit get_pokemon et get_noms_par_type pour qu'elles passent par le cache (mêmes noms : les étapes 2 à 4 en profitent).
def get_pokemon(nom_ou_id):
    identifiant = nettoyer_identifiant(nom_ou_id)
    if identifiant is None:
        return None
    data = requete_avec_cache(f"{BASE_URL}/pokemon/{identifiant}")
    return formater_pokemon(data) if data else None


def get_noms_par_type(nom_type):
    identifiant = nettoyer_identifiant(nom_type)
    if identifiant is None:
        return None
    data = requete_avec_cache(f"{BASE_URL}/type/{identifiant}")
    if data is None:
        return None
    return [p["pokemon"]["name"] for p in data.get("pokemon", [])]

In [33]:
# On redéfinit aussi la version asynchrone (Q2) pour qu'elle consulte le même cache avant d'appeler l'API.
async def get_pokemon_async(session, nom_ou_id, semaphore):
    identifiant = nettoyer_identifiant(nom_ou_id)
    if identifiant is None:
        return None

    url = f"{BASE_URL}/pokemon/{identifiant}"
    data = cache_api.get(url)                       # 1. on regarde d'abord dans le cache
    if data is None:
        data = await requete_robuste_async(session, url, semaphore)   # 2. sinon, appel asynchrone
        if data is not None:
            cache_api.set(url, data)                # 3. on stocke le succès
    return formater_pokemon(data) if data else None

In [34]:
# Démonstration de la rapidité : le 1er appel interroge l'API, le 2e (identique) est servi par le cache.
debut = time.perf_counter()
print(get_pokemon("charizard"))
duree_1 = time.perf_counter() - debut
print(f"⏱️ 1er appel (API)   : {duree_1 * 1000:.1f} ms")

debut = time.perf_counter()
print(get_pokemon("charizard"))
duree_2 = time.perf_counter() - debut
print(f"⏱️ 2e appel (cache)  : {duree_2 * 1000:.3f} ms")

print(f"\n➡️ {duree_1 / duree_2:,.0f} fois plus rapide")

# 5 Pokémon demandés 3 fois chacun : seuls les premiers appels touchent l'API
for _ in range(3):
    for nom in ["pikachu", "bulbasaur", "squirtle", "mewtwo", "eevee"]:
        get_pokemon(nom)
print("\n", cache_api.stats())

{'nom': 'charizard', 'id': 6, 'types': ['fire', 'flying'], 'taille_m': 1.7, 'poids_kg': 90.5, 'stats': {'hp': 78, 'attack': 84, 'defense': 78, 'special-attack': 109, 'special-defense': 85, 'speed': 100}}
⏱️ 1er appel (API)   : 96.1 ms
{'nom': 'charizard', 'id': 6, 'types': ['fire', 'flying'], 'taille_m': 1.7, 'poids_kg': 90.5, 'stats': {'hp': 78, 'attack': 84, 'defense': 78, 'special-attack': 109, 'special-defense': 85, 'speed': 100}}
⏱️ 2e appel (cache)  : 0.147 ms

➡️ 655 fois plus rapide

 11 hits / 6 misses (taux de réussite du cache : 64.7 %)


In [35]:
# Démonstration sur un type complet : le 1er calcul remplit le cache, le 2e est quasi instantané (aucun appel à l'API).
print("===== 1er calcul (cache vide pour ce type) =====")
duree_1 = await stats_type_async("water")

print("\n===== 2e calcul du même type (tout vient du cache) =====")
duree_2 = await stats_type_async("water")

print(f"\n➡️ 2e calcul {duree_1 / duree_2:,.0f} fois plus rapide")
print(cache_api.stats())

===== 1er calcul (cache vide pour ce type) =====
🔥 Nombre de Pokémon de type water : 192
⏳ Récupération en parallèle (20 requêtes simultanées)...

❤️ Moyenne des points de vie : 71.6
   PV minimum : 10 (wiglett)
   PV maximum : 170 (wailord)

⏱️ Temps de calcul : 1.5 s pour 192 requêtes (asynchrone)

===== 2e calcul du même type (tout vient du cache) =====
🔥 Nombre de Pokémon de type water : 192
⏳ Récupération en parallèle (20 requêtes simultanées)...

❤️ Moyenne des points de vie : 71.6
   PV minimum : 10 (wiglett)
   PV maximum : 170 (wailord)

⏱️ Temps de calcul : 0.0 s pour 192 requêtes (asynchrone)

➡️ 2e calcul 414 fois plus rapide
205 hits / 198 misses (taux de réussite du cache : 50.9 %)


In [38]:
# Démonstration de l'expiration avec un cache de test de 3 secondes (on ne va pas attendre 1 heure !).
cache_test = CacheExpirant(duree_vie=3)
url_test = f"{BASE_URL}/pokemon/pikachu"

requete_avec_cache(url_test, cache=cache_test)
print("1. Premier appel         →", cache_test.stats())

requete_avec_cache(url_test, cache=cache_test)
print("2. Appel immédiat        →", cache_test.stats())

print("   ...on attend 4 secondes (plus que la durée de vie)...")
time.sleep(4)

requete_avec_cache(url_test, cache=cache_test)
print("3. Après expiration      →", cache_test.stats())

1. Premier appel         → 0 hits / 1 misses (taux de réussite du cache : 0.0 %)
2. Appel immédiat        → 1 hits / 1 misses (taux de réussite du cache : 50.0 %)
   ...on attend 4 secondes (plus que la durée de vie)...
3. Après expiration      → 1 hits / 2 misses (taux de réussite du cache : 33.3 %)


## 8. Simulation d'un combat en 5 tours
Règles choisies :
- le Pokémon le plus rapide (speed) attaque en premier ;
- chaque Pokémon utilise sa meilleure attaque : physique (attack contre defense)
  ou spéciale (special-attack contre special-defense) ;
- dégâts calculés avec une version simplifiée de la formule officielle des jeux
  (niveau 50, puissance 60) : dégâts = ((2×50/5+2) × 60 × attaque/défense) / 50 + 2 ;
- facteur aléatoire entre 0,85 et 1, et 1 chance sur 16 de coup critique (×1,5) ;
- 5 tours, chacun attaque une fois par tour : le plus grand total de dégâts infligés gagne.

In [39]:
# Calcule les dégâts d'une attaque : choix physique/spéciale, formule simplifiée des jeux, hasard et coup critique.
import random

NIVEAU = 50
PUISSANCE = 60

def calculer_degats(attaquant, defenseur):
    s_att = attaquant["stats"]
    s_def = defenseur["stats"]

    # 1. L'attaquant utilise sa meilleure statistique d'attaque
    if s_att["attack"] >= s_att["special-attack"]:
        categorie, att, defense = "physique", s_att["attack"], s_def["defense"]
    else:
        categorie, att, defense = "spéciale", s_att["special-attack"], s_def["special-defense"]

    # 2. Formule simplifiée des jeux Pokémon
    base = ((2 * NIVEAU / 5 + 2) * PUISSANCE * att / defense) / 50 + 2

    # 3. Un peu de hasard : variation de 85 à 100 %, et 1 chance sur 16 de coup critique
    aleatoire = random.uniform(0.85, 1.0)
    critique = random.random() < 1 / 16

    degats = round(base * aleatoire * (1.5 if critique else 1))
    return degats, categorie, critique

In [40]:
# Déroule un combat de 5 tours entre deux Pokémon et renvoie le résultat (sans rien afficher).
def combat(p1, p2, nb_tours=5, graine=None):

    # Graine aléatoire facultative : permet de rejouer exactement le même combat
    if graine is not None:
        random.seed(graine)

    # Si c'est le même Pokémon des deux côtés, on renomme le second pour ne pas mélanger leurs dégâts
    if p1["nom"] == p2["nom"]:
        p2 = {**p2, "nom": p2["nom"] + " (2)"}

    # Le plus rapide attaque en premier
    if p2["stats"]["speed"] > p1["stats"]["speed"]:
        premier, second = p2, p1
    else:
        premier, second = p1, p2

    totaux = {p1["nom"]: 0, p2["nom"]: 0}
    journal = []

    for tour in range(1, nb_tours + 1):
        for attaquant, defenseur in [(premier, second), (second, premier)]:
            degats, categorie, critique = calculer_degats(attaquant, defenseur)
            totaux[attaquant["nom"]] += degats
            journal.append({
                "tour": tour,
                "attaquant": attaquant["nom"],
                "defenseur": defenseur["nom"],
                "degats": degats,
                "categorie": categorie,
                "critique": critique,
            })

    # Le plus grand total de dégâts gagne (None = égalité)
    if totaux[p1["nom"]] > totaux[p2["nom"]]:
        vainqueur = p1["nom"]
    elif totaux[p2["nom"]] > totaux[p1["nom"]]:
        vainqueur = p2["nom"]
    else:
        vainqueur = None

    return {"premier": premier["nom"], "totaux": totaux, "journal": journal, "vainqueur": vainqueur}

In [41]:
# Demande deux Pokémon à l'utilisateur, lance le combat et raconte son déroulement tour par tour.
def simuler_combat():
    choix1 = input("Premier Pokémon : ")
    choix2 = input("Deuxième Pokémon : ")

    p1 = get_pokemon(choix1)
    p2 = get_pokemon(choix2)
    if p1 is None or p2 is None:
        print("❌ Impossible de lancer le combat : un des Pokémon est introuvable.")
        return

    resultat = combat(p1, p2)

    print(f"\n⚔️  {p1['nom'].capitalize()} contre {p2['nom'].capitalize()} — 5 tours")
    print(f"💨 {resultat['premier'].capitalize()} est le plus rapide : il attaque en premier.\n")

    tour_actuel = 0
    for attaque in resultat["journal"]:
        if attaque["tour"] != tour_actuel:
            tour_actuel = attaque["tour"]
            print(f"--- Tour {tour_actuel} ---")
        texte_critique = "  💥 COUP CRITIQUE !" if attaque["critique"] else ""
        print(f"  {attaque['attaquant'].capitalize()} lance une attaque {attaque['categorie']} "
              f"sur {attaque['defenseur'].capitalize()} : {attaque['degats']} dégâts{texte_critique}")

    print("\n📊 Dégâts totaux infligés :")
    for nom, total in resultat["totaux"].items():
        print(f"  {nom.capitalize():<15} {total}")

    if resultat["vainqueur"]:
        print(f"\n🏆 {resultat['vainqueur'].capitalize()} remporte le combat !")
    else:
        print("\n🤝 Égalité parfaite !")


simuler_combat()

Premier Pokémon : pikachu
Deuxième Pokémon : snorlax

⚔️  Pikachu contre Snorlax — 5 tours
💨 Pikachu est le plus rapide : il attaque en premier.

--- Tour 1 ---
  Pikachu lance une attaque physique sur Snorlax : 24 dégâts
  Snorlax lance une attaque physique sur Pikachu : 72 dégâts
--- Tour 2 ---
  Pikachu lance une attaque physique sur Snorlax : 22 dégâts
  Snorlax lance une attaque physique sur Pikachu : 64 dégâts
--- Tour 3 ---
  Pikachu lance une attaque physique sur Snorlax : 24 dégâts
  Snorlax lance une attaque physique sur Pikachu : 64 dégâts
--- Tour 4 ---
  Pikachu lance une attaque physique sur Snorlax : 24 dégâts
  Snorlax lance une attaque physique sur Pikachu : 71 dégâts
--- Tour 5 ---
  Pikachu lance une attaque physique sur Snorlax : 23 dégâts
  Snorlax lance une attaque physique sur Pikachu : 72 dégâts

📊 Dégâts totaux infligés :
  Pikachu         117
  Snorlax         343

🏆 Snorlax remporte le combat !


In [42]:
# On rejoue 100 fois le même duel pour voir si le hasard peut changer le vainqueur (statistiques de victoires).
from collections import Counter

def statistiques_duel(nom1, nom2, nb_combats=100):
    p1, p2 = get_pokemon(nom1), get_pokemon(nom2)
    victoires = Counter(combat(p1, p2)["vainqueur"] or "égalité" for _ in range(nb_combats))
    print(f"Sur {nb_combats} combats {nom1} contre {nom2} :")
    for nom, nb in victoires.most_common():
        print(f"  {nom:<15} {nb} victoires")

statistiques_duel("charizard", "pikachu")
print()
statistiques_duel("snorlax", "machamp")

Sur 100 combats charizard contre pikachu :
  charizard       100 victoires

Sur 100 combats snorlax contre machamp :
  machamp         100 victoires


## 9. Interface Streamlit
Application web qui présente les retours de la PokéAPI en 4 onglets :
statistiques d'un Pokémon, comparaison de deux Pokémon, statistiques d'un type et combat.
Les appels réutilisent la gestion d'erreurs (Q1) et un cache d'1 heure (Q3, via st.cache_data).
Pour les types, les Pokémon sont récupérés en parallèle avec des threads (ThreadPoolExecutor),
une alternative à asyncio plus adaptée à Streamlit.
L'application est exposée depuis Colab grâce à un tunnel Cloudflare.

In [43]:
# On installe Streamlit, la bibliothèque qui transforme un script Python en site web interactif.
!pip install -q streamlit

In [44]:
%%writefile app_pokemon.py
# Application Streamlit qui présente les retours de la PokéAPI : stats, comparaison, types et combat.
import random
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
import requests
import streamlit as st

BASE_URL = "https://pokeapi.co/api/v2"
CODES_TEMPORAIRES = {429, 500, 502, 503, 504}
MESSAGES_ERREURS = {400: "Requête mal formée (400)", 401: "Non autorisé (401)",
                    403: "Accès interdit (403)", 404: "Introuvable (404)"}
NOMS_STATS = {"hp": "Points de vie", "attack": "Attaque", "defense": "Défense",
              "special-attack": "Attaque spéciale", "special-defense": "Défense spéciale",
              "speed": "Vitesse"}
NIVEAU, PUISSANCE = 50, 60

st.set_page_config(page_title="Pokédex API", page_icon="⚡", layout="wide")


# ---------- Appels API : robustes (Q1) + cache d'1 heure (Q3) ----------
@st.cache_data(ttl=3600, show_spinner=False)
def requete_robuste(url, max_tentatives=5):
    """Retry + backoff sur les erreurs temporaires ; exception (non mise en cache) sinon."""
    for tentative in range(1, max_tentatives + 1):
        attente = 2 ** (tentative - 1)
        try:
            response = requests.get(url, timeout=10)
        except requests.exceptions.RequestException:
            time.sleep(attente)
            continue
        if response.status_code == 200:
            return response.json()
        if response.status_code in CODES_TEMPORAIRES:
            time.sleep(attente)
            continue
        raise RuntimeError(MESSAGES_ERREURS.get(response.status_code, f"Erreur {response.status_code}"))
    raise RuntimeError("Échec après plusieurs tentatives")


def get_pokemon(nom_ou_id):
    """Renvoie un dictionnaire propre, ou None si introuvable."""
    identifiant = str(nom_ou_id).strip().lower()
    if not identifiant:
        return None
    try:
        data = requete_robuste(f"{BASE_URL}/pokemon/{identifiant}")
    except RuntimeError:
        return None
    images = data.get("sprites", {})
    artwork = images.get("other", {}).get("official-artwork", {}).get("front_default")
    return {
        "nom": data["name"],
        "id": data["id"],
        "types": [t["type"]["name"] for t in data.get("types", [])],
        "taille_m": data.get("height", 0) / 10,
        "poids_kg": data.get("weight", 0) / 10,
        "image": artwork or images.get("front_default"),
        "stats": {s["stat"]["name"]: s["base_stat"] for s in data.get("stats", [])},
    }


# ---------- Combat ----------
def calculer_degats(attaquant, defenseur):
    s_att, s_def = attaquant["stats"], defenseur["stats"]
    if s_att["attack"] >= s_att["special-attack"]:
        categorie, att, defense = "physique", s_att["attack"], s_def["defense"]
    else:
        categorie, att, defense = "spéciale", s_att["special-attack"], s_def["special-defense"]
    base = ((2 * NIVEAU / 5 + 2) * PUISSANCE * att / defense) / 50 + 2
    critique = random.random() < 1 / 16
    degats = round(base * random.uniform(0.85, 1.0) * (1.5 if critique else 1))
    return degats, categorie, critique


def combat(p1, p2, nb_tours=5):
    if p1["nom"] == p2["nom"]:
        p2 = {**p2, "nom": p2["nom"] + " (2)"}
    premier, second = (p2, p1) if p2["stats"]["speed"] > p1["stats"]["speed"] else (p1, p2)
    totaux = {p1["nom"]: 0, p2["nom"]: 0}
    journal = []
    for tour in range(1, nb_tours + 1):
        for attaquant, defenseur in [(premier, second), (second, premier)]:
            degats, categorie, critique = calculer_degats(attaquant, defenseur)
            totaux[attaquant["nom"]] += degats
            journal.append({"Tour": tour, "Attaquant": attaquant["nom"].capitalize(),
                            "Cible": defenseur["nom"].capitalize(), "Attaque": categorie,
                            "Dégâts": degats, "Critique": "💥" if critique else ""})
    t1, t2 = totaux[p1["nom"]], totaux[p2["nom"]]
    vainqueur = p1["nom"] if t1 > t2 else p2["nom"] if t2 > t1 else None
    return {"premier": premier["nom"], "totaux": totaux, "journal": journal, "vainqueur": vainqueur}


# ---------- Petits composants d'affichage ----------
def carte_pokemon(p):
    if p["image"]:
        st.image(p["image"], width=180)
    st.subheader(f"{p['nom'].capitalize()} (n°{p['id']})")
    st.write(f"**Type(s) :** {', '.join(p['types'])}")
    st.write(f"**Taille :** {p['taille_m']} m · **Poids :** {p['poids_kg']} kg")


def tableau_stats(p):
    return pd.DataFrame({"Statistique": [NOMS_STATS[k] for k in p["stats"]],
                         "Valeur": list(p["stats"].values())})


# ---------- Interface ----------
st.title("⚡ Pokédex — PokéAPI")
st.caption("Appels robustes (retry) et mis en cache 1 h · données : pokeapi.co")

onglet_stats, onglet_comp, onglet_type, onglet_combat = st.tabs(
    ["📊 Statistiques", "⚖️ Comparer", "🔥 Par type", "⚔️ Combat"]
)

# Onglet 1 : statistiques d'un Pokémon
with onglet_stats:
    choix = st.text_input("Nom (en anglais) ou numéro du Pokémon", value="pikachu")
    p = get_pokemon(choix)
    if p is None:
        st.error(f"❌ Pokémon « {choix} » introuvable.")
    else:
        col1, col2 = st.columns([1, 2])
        with col1:
            carte_pokemon(p)
            st.metric("Total des statistiques", sum(p["stats"].values()))
        with col2:
            st.bar_chart(tableau_stats(p), x="Statistique", y="Valeur", horizontal=True)

# Onglet 2 : comparaison de deux Pokémon
with onglet_comp:
    c1, c2 = st.columns(2)
    nom1 = c1.text_input("Premier Pokémon", value="charizard")
    nom2 = c2.text_input("Deuxième Pokémon", value="pikachu")
    p1, p2 = get_pokemon(nom1), get_pokemon(nom2)
    if p1 is None or p2 is None:
        st.error("❌ Un des Pokémon est introuvable.")
    else:
        with c1:
            carte_pokemon(p1)
        with c2:
            carte_pokemon(p2)

        comparaison = pd.DataFrame({
            "Statistique": [NOMS_STATS[k] for k in p1["stats"]],
            p1["nom"].capitalize(): list(p1["stats"].values()),
            p2["nom"].capitalize() + (" (2)" if p1["nom"] == p2["nom"] else ""): list(p2["stats"].values()),
        })
        st.bar_chart(comparaison, x="Statistique", stack=False)
        st.dataframe(comparaison, hide_index=True, use_container_width=True)

        def verdict(stat):
            v1, v2 = p1["stats"][stat], p2["stats"][stat]
            return p1["nom"].capitalize() if v1 > v2 else p2["nom"].capitalize() if v2 > v1 else "Égalité"

        st.success(f"🏆 Plus fort en points de vie : **{verdict('hp')}** · "
                   f"Plus fort en attaque : **{verdict('attack')}**")

# Onglet 3 : statistiques d'un type
with onglet_type:
    types = ["normal", "fire", "water", "grass", "electric", "ice", "fighting", "poison", "ground",
             "flying", "psychic", "bug", "rock", "ghost", "dragon", "dark", "steel", "fairy"]
    nom_type = st.selectbox("Type", types, index=types.index("fire"))
    try:
        noms = [x["pokemon"]["name"] for x in requete_robuste(f"{BASE_URL}/type/{nom_type}")["pokemon"]]
    except RuntimeError as erreur:
        st.error(f"❌ {erreur}")
        noms = []

    if noms:
        debut = time.perf_counter()
        with st.spinner(f"Récupération des {len(noms)} Pokémon en parallèle..."):
            with ThreadPoolExecutor(max_workers=20) as executeur:
                pokemons = [p for p in executeur.map(get_pokemon, noms) if p]
        duree = time.perf_counter() - debut

        df_type = pd.DataFrame({"Pokémon": [p["nom"] for p in pokemons],
                                "PV": [p["stats"]["hp"] for p in pokemons],
                                "Attaque": [p["stats"]["attack"] for p in pokemons]})

        m1, m2, m3, m4 = st.columns(4)
        m1.metric("Nombre de Pokémon", len(noms))
        m2.metric("Moyenne des PV", f"{df_type['PV'].mean():.1f}")
        m3.metric("PV max", f"{df_type['PV'].max()} ({df_type.loc[df_type['PV'].idxmax(), 'Pokémon']})")
        m4.metric("Temps de calcul", f"{duree:.2f} s")

        st.subheader("Les 15 Pokémon avec le plus de PV")
        top = df_type.sort_values("PV", ascending=False).head(15)
        st.bar_chart(top, x="Pokémon", y="PV", horizontal=True)
        st.dataframe(df_type.sort_values("PV", ascending=False), hide_index=True, use_container_width=True)

# Onglet 4 : combat
with onglet_combat:
    c1, c2 = st.columns(2)
    combattant1 = c1.text_input("Combattant 1", value="charizard", key="cb1")
    combattant2 = c2.text_input("Combattant 2", value="blastoise", key="cb2")

    if st.button("⚔️ Lancer le combat", type="primary"):
        a, b = get_pokemon(combattant1), get_pokemon(combattant2)
        if a is None or b is None:
            st.error("❌ Un des Pokémon est introuvable.")
        else:
            resultat = combat(a, b)
            with c1:
                carte_pokemon(a)
            with c2:
                carte_pokemon(b)

            st.info(f"💨 {resultat['premier'].capitalize()} est le plus rapide : il attaque en premier.")

            journal = pd.DataFrame(resultat["journal"])
            cumul = (journal.pivot_table(index="Tour", columns="Attaquant", values="Dégâts", aggfunc="sum")
                     .cumsum())
            st.subheader("Dégâts cumulés tour après tour")
            st.line_chart(cumul)

            t1, t2 = st.columns(2)
            noms_totaux = list(resultat["totaux"].items())
            t1.metric(f"Dégâts infligés par {noms_totaux[0][0].capitalize()}", noms_totaux[0][1])
            t2.metric(f"Dégâts infligés par {noms_totaux[1][0].capitalize()}", noms_totaux[1][1])

            if resultat["vainqueur"]:
                st.success(f"🏆 **{resultat['vainqueur'].capitalize()}** remporte le combat !")
                st.balloons()
            else:
                st.warning("🤝 Égalité parfaite !")

            with st.expander("Détail des 10 attaques"):
                st.dataframe(journal, hide_index=True, use_container_width=True)

Writing app_pokemon.py


In [45]:
# On lance Streamlit en arrière-plan (indépendamment de la cellule) puis on vérifie qu'il répond bien (code 200).
import time

!pkill -f "streamlit run"
!nohup streamlit run app_pokemon.py --server.port 8501 --server.headless true > /content/logs.txt 2>&1 &

time.sleep(6)
print("Réponse de Streamlit (200 = OK) :")
!curl -s -o /dev/null -w "%{http_code}" http://localhost:8501

Réponse de Streamlit (200 = OK) :
200

In [46]:
# On ouvre un tunnel Cloudflare pour rendre l'application accessible depuis ton navigateur, puis on affiche son adresse.
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!nohup ./cloudflared tunnel --url http://localhost:8501 > cloudflared.log 2>&1 &

time.sleep(8)
print("👉 Ouvre ce lien :")
!grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' cloudflared.log | head -1

👉 Ouvre ce lien :
https://bumper-css-league-those.trycloudflare.com
